In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from huggingface_hub import login
login()   # dán token Hugging Face (Read là đủ)

!pip -q install gradio librosa sacremoses
import gradio as gr, torch, numpy as np, librosa
from transformers import (pipeline, MarianMTModel, MarianTokenizer,
                          AutoProcessor, SeamlessM4Tv2ForSpeechToText)
SR, MAX_SEC = 16000, 30

asr = pipeline("automatic-speech-recognition", model="vinai/PhoWhisper-small",
               device=0, dtype=torch.float16)
gk = {"language": "vi", "task": "transcribe", "num_beams": 3}

def load_mt(name):
    return (MarianTokenizer.from_pretrained(name),
            MarianMTModel.from_pretrained(name).to("cuda").eval())
A = load_mt("Helsinki-NLP/opus-mt-vi-en")
B = load_mt("trthDux/opus-mt-vi-en-asr-ft")

sproc = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
smodel = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    "facebook/seamless-m4t-v2-large", dtype=torch.float16).to("cuda").eval()

def mt(pair, text):
    t, m = pair
    x = t(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        y = m.generate(**x, num_beams=4, max_new_tokens=160)
    return t.decode(y[0], skip_special_tokens=True)

def seamless(wav):
    f = sproc(audio=[wav], sampling_rate=SR, return_tensors="pt", padding=True).to("cuda")
    f["input_features"] = f["input_features"].half()
    with torch.no_grad():
        out = smodel.generate(**f, tgt_lang="eng", num_beams=4, max_new_tokens=160)
    if hasattr(out, "sequences"): out = out.sequences
    return sproc.batch_decode(out, skip_special_tokens=True)[0].strip()

def run(path):
    if path is None: return "", "", "", ""
    wav, _ = librosa.load(path, sr=SR, mono=True)
    wav = wav[: SR * MAX_SEC].astype("float32")
    vi = asr({"raw": wav, "sampling_rate": SR}, generate_kwargs=gk)["text"].strip()
    return vi, mt(A, vi), mt(B, vi), seamless(wav)

In [ ]:
demo = gr.Interface(
    fn=run,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath",
                    label="Nói tiếng Việt (tối đa 30 giây)"),
    outputs=[gr.Textbox(label="ASR tiếng Việt (PhoWhisper)"),
             gr.Textbox(label="A: OPUS-MT gốc"),
             gr.Textbox(label="B: OPUS-MT fine-tune"),
             gr.Textbox(label="C: Seamless (trực tiếp từ audio)")],
    title="Demo vi → en: A, B, C",
)
demo.launch(share=True)